# Nested Loops and Common Patterns

**Topic:** `2.4` &nbsp;|&nbsp; **Module 2: Control Flow and Loops**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M2)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [02_control_flow/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Count the cells of a grid

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** nested loops, counting, ragged data

**Explanation and reasoning**

Iterating the rows themselves is what makes this correct for a ragged grid: the inner loop's length comes from the row in front of it, so no cell is missed and none is invented. A hard-coded width would count a short row incorrectly or raise on it.

In [ ]:
def count_cells(grid) -> int:
    total = 0
    for row in grid:
        for cell in row:
            total += 1
    return total

**Complexity**

Time O(total cells); space O(1).

**Edge cases and failure modes**

A grid of empty rows gives 0, because no inner iteration runs.

**Alternative approaches**

sum(len(row) for row in grid) is the same answer in one line.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert count_cells(((0, 1), (0,), (1, 1, 1))) == 6
assert count_cells(()) == 0

---

## Solution 2 — Sum every cell with a nested loop

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** nested loops, accumulators

**Explanation and reasoning**

The continue keeps the accumulation on one unindented line, which is the reason to prefer it over an else block here. Rejecting bool matters because True would otherwise contribute 1 to a physical total.

In [ ]:
def sum_grid(grid) -> float:
    total = 0.0
    for row in grid:
        for cell in row:
            if isinstance(cell, bool) or not isinstance(cell, (int, float)):
                continue
            total += cell
    return total

**Complexity**

Time O(total cells); space O(1).

**Edge cases and failure modes**

A grid of only junk returns 0.0 rather than raising.

**Alternative approaches**

A nested comprehension is shorter but hides the skip rule.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert sum_grid(((1, 2), (3, 'x'), ())) == 6
assert sum_grid(()) == 0.0

---

## Solution 3 — Enumerate every coordinate

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** enumerate, generators, nested loops

**Explanation and reasoning**

A generator function is itself an iterator, so the nested yields produce the pairs lazily and list() materialises them at the end. Using enumerate at both levels means the coordinates are computed rather than tracked in a counter.

In [ ]:
def coordinates(grid):
    for row_index, row in enumerate(grid):
        for col_index, cell in enumerate(row):
            yield row_index, col_index

**Complexity**

Time O(total cells); space O(1) before list() consumes it.

**Edge cases and failure modes**

An empty grid yields nothing, so list() gives an empty list.

**Alternative approaches**

A nested comprehension is the eager equivalent.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert list(coordinates(((0, 1), (2,)))) == [(0, 0), (0, 1), (1, 0)]
assert list(coordinates(())) == []

---

## Solution 4 — Sliding window of fixed size

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** slicing, range, single pass

**Explanation and reasoning**

Stepping the start index by one and slicing a window gives every window in a single pass, where the nested version would re-scan the whole prefix for each start. The range bound of len - size + 1 is what makes a size larger than the data produce no windows rather than a short one.

In [ ]:
def windows(values, size: int) -> list[tuple]:
    if not isinstance(size, int) or isinstance(size, bool) or size < 1:
        raise ValueError(f'size must be a positive integer, got {size!r}')
    return [
        tuple(values[start:start + size])
        for start in range(len(values) - size + 1)
    ]

**Complexity**

Time O(n * size) for the copies; space O(n * size).

**Edge cases and failure modes**

A size equal to the length gives exactly one window.

**Alternative approaches**

collections.deque(maxlen=size) gives O(n) with a bounded buffer.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert windows((1, 2, 3, 4), 2) == [(1, 2), (2, 3), (3, 4)]
assert windows((1, 2), 5) == []

---

## Solution 5 — Derive the pairwise product sum

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** algebra, nested loops, optimisation

**Explanation and reasoning**

Expanding the square of a sum produces exactly the cross terms the double loop accumulates, so the identity gives the same number in O(n). Keeping the naive version makes the equivalence testable rather than asserted.

In [ ]:
def pair_product_sum(values):
    total = sum(values)
    return total * total


def pair_product_sum_naive(values):
    """The O(n^2) reference, kept only for the tests."""
    total = 0
    for a in values:
        for b in values:
            total += a * b
    return total

**Complexity**

Time O(n) for the fast version, O(n^2) for the naive one.

**Edge cases and failure modes**

An empty sequence sums to 0, so the square is 0.

**Alternative approaches**

math.fsum reduces float error when the values are large.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert pair_product_sum((1, 2, 3)) == 36
assert pair_product_sum(()) == 0
assert pair_product_sum((1, -2, 3)) == pair_product_sum_naive((1, -2, 3))

---

## Solution 6 — Find the first occupied cell

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** nested loops, return, early exit

**Explanation and reasoning**

Returning from inside the inner loop leaves both loops at once, which a break could not do - it would only end the inner loop and let the outer one carry on. Iterating the rows keeps the search safe on a ragged grid.

In [ ]:
def first_occupied(grid):
    for row_index, row in enumerate(grid):
        for col_index, cell in enumerate(row):
            if cell:
                return row_index, col_index
    return None

**Complexity**

Time O(total cells) worst case; space O(1).

**Edge cases and failure modes**

A cell of 0 is falsy, so it is not treated as occupied.

**Alternative approaches**

next() over a coordinate generator is the lazy form.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert first_occupied(((0, 0), (0, 1))) == (1, 1)
assert first_occupied(()) is None

---

## Solution 7 — Count row and column totals

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** nested loops, aggregation, ragged data

**Explanation and reasoning**

Tracking the maximum row length as the scan proceeds is what makes the column count true for a ragged grid, where len(grid[0]) would understate it and range(len(grid[0])) would miss cells entirely.

In [ ]:
def grid_totals(grid) -> dict:
    rows = 0
    cols = 0
    total = 0.0
    for row in grid:
        rows += 1
        if len(row) > cols:
            cols = len(row)
        for cell in row:
            total += cell
    return {'rows': rows, 'cols': cols, 'sum': total}

**Complexity**

Time O(total cells); space O(1).

**Edge cases and failure modes**

A grid of empty rows gives cols 0 and a sum of 0.0.

**Alternative approaches**

Per-row sums plus a max() give the same answer.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert grid_totals(((1, 2), (3,))) == {'rows': 2, 'cols': 2, 'sum': 6.0}
assert grid_totals(()) == {'rows': 0, 'cols': 0, 'sum': 0.0}

---

## Solution 8 — Find the strongest neighbour

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** nested loops, bounds checks, max selection

**Explanation and reasoning**

The bounds check must come before the indexing, because the offset loop deliberately walks off the edge of the grid. Seeding best_value with None rather than 0 is what lets a grid of all-negative values still produce a result.

In [ ]:
def strongest_neighbour(grid, row, col):
    best = None
    best_value = None
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0:
                continue
            r, c = row + dr, col + dc
            if not (0 <= r < len(grid) and 0 <= c < len(grid[r])):
                continue
            value = grid[r][c]
            if best_value is None or value > best_value:
                best_value = value
                best = (dr, dc)
    return best

**Complexity**

Time O(9) regardless of grid size; space O(1).

**Edge cases and failure modes**

A one-cell grid has no in-bounds neighbours and returns None.

**Alternative approaches**

max() with a generator over the offsets is the one-liner.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert strongest_neighbour(((1, 9), (0, 0)), 0, 0) == (0, 1)
assert strongest_neighbour(((5,),), 0, 0) is None

---

## Solution 9 — Check survey coverage against a threshold

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** nested loops, division, validation

**Explanation and reasoning**

Counting total inside the loop is what makes the denominator honest for a ragged grid, and the conditional expression is what keeps an empty grid from dividing by zero. Reporting zero rather than a perfect 1.0 is the honest answer for a survey of nothing.

In [ ]:
def coverage(grid, is_covered, min_ratio: float = 0.5) -> dict:
    total = 0
    covered = 0
    for row in grid:
        for cell in row:
            total += 1
            if is_covered(cell):
                covered += 1
    ratio = (covered / total) if total else 0.0
    return {
        'total': total,
        'covered': covered,
        'ratio': round(ratio, 3),
        'ok': ratio >= min_ratio,
    }

**Complexity**

Time O(total cells); space O(1).

**Edge cases and failure modes**

An empty grid gives ratio 0.0 and ok False.

**Alternative approaches**

Returning the uncovered coordinates tells the caller where to go.

**Tests — run the cell to verify the reference solution**

In [ ]:
out = coverage(((1, 0), (1, 0)), lambda c: c == 1)
assert out['ratio'] == 0.5 and out['ok'] is True
assert coverage((), lambda c: True)['ratio'] == 0.0

---

## Solution 10 — Plan a route across a terrain grid

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** nested loops, simulator API, planning

**Explanation and reasoning**

Reading the battery once before the scan keeps the API call out of the inner loop and makes the depleted case a single early return rather than a condition repeated per cell. The nested scan then only classifies terrain, which is the part that genuinely needs two levels.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

def plan_cells(grid, robot, min_battery_pct: float = 20.0) -> list[tuple]:
    """Return passable cells in row-major order."""
    battery = robot.status()['battery_pct']
    if battery <= min_battery_pct:
        return []
    plan: list[tuple] = []
    for row_index, row in enumerate(grid):
        for col_index, cell in enumerate(row):
            if cell == 0:
                plan.append((row_index, col_index))
    return plan

**Complexity**

Time O(total cells); space O(passable cells).

**Edge cases and failure modes**

A depleted robot returns an empty list rather than a partial plan.

**Alternative approaches**

A comprehension over enumerate produces the same list more tersely.

**Tests — run the cell to verify the reference solution**

In [ ]:
robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
assert plan_cells(((0, 1), (0, 0)), robot) == [(0, 0), (1, 0), (1, 1)]
assert plan_cells((), robot) == []

---

# Robotics challenge solution — ROBO-X Challenge: Survey Grid Planner

**Explanation**

The nested scan is used only where two levels are genuinely needed, and the battery is read once so the API call stays out of the inner loop. The readiness verdict combines two independent conditions, so a high coverage ratio cannot mask a flat battery.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

def survey_coverage(robot, grid, min_ratio: float = 0.5) -> dict:
    """Combine a live battery reading with a grid coverage check."""
    battery = robot.status()['battery_pct']
    total = 0
    covered = 0
    for row_index, row in enumerate(grid):
        for col_index, cell in enumerate(row):
            total += 1
            if cell:
                covered += 1
    ratio = (covered / total) if total else 0.0
    ready = ratio >= min_ratio and battery > min_ratio
    return {
        'battery_pct': round(battery, 2),
        'total': total,
        'covered': covered,
        'ratio': round(ratio, 3),
        'ready': ready,
    }

**Complexity**

Time O(total cells); space O(1).

**Notes and trade-offs**

An empty grid gives ratio 0.0, which fails the coverage test.

In [ ]:
robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
out = survey_coverage(robot, ((1, 1), (1, 0)))
assert out['ready'] is True
assert out['total'] == 4

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.